# Building Neural Networks from Scratch

**What you'll learn in this notebook:**
- How `nn.Module` works — building your first neural network
- Parameters vs Buffers — what's tracked and what's not
- A tour of common layers: Linear, Conv2d, BatchNorm, LayerNorm, Dropout
- Building a CNN for image classification step by step
- Building an MLP with modern best practices
- Activation functions compared and visualized
- Loss functions — when to use CrossEntropy, MSE, or BCE
- Saving and loading models the right way

**Prerequisites:** Notebooks 01-02 (Tensors, Autograd).  
**Time:** ~50 minutes  
**Hardware:** CPU only — no GPU required!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
print(f"PyTorch version: {torch.__version__}")

## 1. Your First nn.Module

Every neural network in PyTorch inherits from `nn.Module`. Here's the pattern:

1. Define layers in `__init__` (stored as attributes)
2. Define the forward pass in `forward()`
3. PyTorch handles backward automatically via autograd

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()               # always call super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # layer 1
        self.fc2 = nn.Linear(hidden_size, num_classes)  # layer 2

    def forward(self, x):
        x = self.fc1(x)           # linear transform
        x = F.relu(x)             # activation
        x = self.fc2(x)           # output layer
        return x

model = SimpleNet(input_size=784, hidden_size=128, num_classes=10)
print(model)

# Test with a random batch
x = torch.randn(4, 784)  # batch of 4, 784 features (like flattened 28x28 images)
output = model(x)
print(f"\nInput shape:  {list(x.shape)}")
print(f"Output shape: {list(output.shape)}")
print(f"Output (logits for 10 classes):\n{output}")

### Inspecting Parameters

`nn.Module` automatically tracks all `nn.Parameter` objects as learnable parameters:

In [ ]:
print("=== Named Parameters ===")
total_params = 0
for name, param in model.named_parameters():
    print(f"  {name:15s} shape={str(list(param.shape)):15s} params={param.numel():,}")
    total_params += param.numel()
print(f"  {'TOTAL':15s} {' ':15s} params={total_params:,}")

print(f"\n=== Quick count ===")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 2. Parameters vs Buffers

- **Parameters** (`nn.Parameter`): learnable, updated by optimizer, included in `state_dict`
- **Buffers** (`register_buffer`): not learnable, but part of model state (e.g., running mean in BatchNorm)
- **Regular attributes**: not saved, not moved with `.to(device)`

In [ ]:
class ModelWithBuffer(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("running_mean", torch.zeros(2))  # buffer
        self.step_count = 0  # regular attribute (NOT saved!)

    def forward(self, x):
        out = self.linear(x)
        if self.training:
            self.running_mean = 0.9 * self.running_mean + 0.1 * out.mean(dim=0).detach()
            self.step_count += 1
        return out

m = ModelWithBuffer()
m(torch.randn(8, 4))

print("=== Parameters (learnable) ===")
for name, p in m.named_parameters():
    print(f"  {name}: shape={list(p.shape)}, requires_grad={p.requires_grad}")

print("\n=== Buffers (not learnable, but saved) ===")
for name, b in m.named_buffers():
    print(f"  {name}: shape={list(b.shape)}, requires_grad={b.requires_grad}")

print(f"\n=== state_dict (what gets saved) ===")
for k, v in m.state_dict().items():
    print(f"  {k}: {list(v.shape)}")
print(f"\nstep_count={m.step_count} (NOT in state_dict — would be lost on load!)")

## 3. Common Layers Tour

Let's explore the most-used layers with concrete shape examples.

In [ ]:
batch = 4

print("=== nn.Linear (fully connected) ===")
linear = nn.Linear(in_features=10, out_features=5)
x = torch.randn(batch, 10)
print(f"  Input:  {list(x.shape)} → Output: {list(linear(x).shape)}")
print(f"  Weight: {list(linear.weight.shape)}, Bias: {list(linear.bias.shape)}")

print("\n=== nn.Conv2d ===")
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
x = torch.randn(batch, 3, 32, 32)
print(f"  Input:  {list(x.shape)} → Output: {list(conv(x).shape)}")
print(f"  Weight: {list(conv.weight.shape)} = (out_ch, in_ch, kH, kW)")

print("\n=== nn.BatchNorm2d (normalizes per channel) ===")
bn = nn.BatchNorm2d(num_features=16)
x = torch.randn(batch, 16, 8, 8)
print(f"  Input:  {list(x.shape)} → Output: {list(bn(x).shape)}")

print("\n=== nn.LayerNorm (normalizes per sample) ===")
ln = nn.LayerNorm(normalized_shape=64)
x = torch.randn(batch, 10, 64)
print(f"  Input:  {list(x.shape)} → Output: {list(ln(x).shape)}")

print("\n=== nn.Dropout (randomly zeros elements) ===")
drop = nn.Dropout(p=0.5)
x = torch.ones(1, 10)
drop.train()
print(f"  Training mode: {drop(x)}  (some zeroed, rest scaled by 1/(1-p))")
drop.eval()
print(f"  Eval mode:     {drop(x)}  (identity — no dropout)")

## 4. Building a CNN

Let's build a convolutional neural network step by step for image classification.
Each convolutional block: **Conv2d → BatchNorm → ReLU → MaxPool**

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),    # 28x28 → 28x28
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                                 # 28x28 → 14x14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),   # 14x14 → 14x14
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                                 # 14x14 → 7x7

            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # 7x7 → 7x7
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                         # 7x7 → 1x1
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn = SimpleCNN(num_classes=10)
x = torch.randn(2, 1, 28, 28)  # 2 grayscale 28x28 images
out = cnn(x)
print(f"Input:  {list(x.shape)}")
print(f"Output: {list(out.shape)}")
print(f"\nModel:\n{cnn}")

### Tracing shapes through the network

Let's watch the shape transform at each layer:

In [ ]:
x = torch.randn(1, 1, 28, 28)
print(f"{'Layer':<30s} {'Output Shape':>20s}")
print("-" * 52)
print(f"{'Input':<30s} {str(list(x.shape)):>20s}")

for name, layer in cnn.features.named_children():
    x = layer(x)
    print(f"{name + ': ' + layer.__class__.__name__:<30s} {str(list(x.shape)):>20s}")

for name, layer in cnn.classifier.named_children():
    x = layer(x)
    print(f"{'classifier.' + name + ': ' + layer.__class__.__name__:<30s} {str(list(x.shape)):>20s}")

## 5. Building an MLP with Best Practices

A modern MLP with: initialization, residual connections, and layer normalization.

In [ ]:
class MLPBlock(nn.Module):
    def __init__(self, dim, expansion=4, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim * expansion)
        self.fc2 = nn.Linear(dim * expansion, dim)
        self.drop = nn.Dropout(dropout)
        self._init_weights()

    def _init_weights(self):
        nn.init.kaiming_normal_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_normal_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x + residual  # residual connection

class ModernMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_blocks=3):
        super().__init__()
        self.embed = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.Sequential(*[MLPBlock(hidden_dim) for _ in range(num_blocks)])
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        x = self.embed(x)
        x = self.blocks(x)
        x = self.head(x)
        return x

mlp = ModernMLP(input_dim=784, hidden_dim=256, num_classes=10)
out = mlp(torch.randn(4, 784))
print(f"Output shape: {list(out.shape)}")
print(f"Total params: {sum(p.numel() for p in mlp.parameters()):,}")

## 6. Activation Functions Compared

Activation functions introduce non-linearity. Here are the most common ones, visualized:

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

x = torch.linspace(-4, 4, 200)
activations = {
    "ReLU": F.relu(x),
    "GELU": F.gelu(x),
    "SiLU (Swish)": F.silu(x),
    "Sigmoid": torch.sigmoid(x),
    "Tanh": torch.tanh(x),
}

fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ax, (name, y) in zip(axes, activations.items()):
    ax.plot(x.numpy(), y.numpy(), linewidth=2)
    ax.set_title(name, fontsize=14)
    ax.axhline(y=0, color="k", linewidth=0.5)
    ax.axvline(x=0, color="k", linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-4, 4)

plt.tight_layout()
plt.savefig("/tmp/pytorch-reference-guide/notebooks/activations.png", dpi=100, bbox_inches="tight")
plt.show()
print("Plot saved to activations.png")

print("\nWhen to use what:")
print("  ReLU:    Simple, fast, default choice for CNNs")
print("  GELU:    Default in transformers (BERT, GPT)")
print("  SiLU:    Popular in modern architectures (EfficientNet)")
print("  Sigmoid: Output layer for binary classification")
print("  Tanh:    Output layer when you need [-1, 1] range")

## 7. Loss Functions

The loss function measures how far your predictions are from the targets. Different tasks need different losses.

| Loss | Task | Input | Target |
|------|------|-------|--------|
| `CrossEntropyLoss` | Multi-class classification | Logits `(B, C)` | Class indices `(B,)` |
| `MSELoss` | Regression | Any shape | Same shape |
| `BCEWithLogitsLoss` | Binary/multi-label classification | Logits `(B, C)` | Float `(B, C)` in [0,1] |

In [ ]:
print("=== CrossEntropyLoss (multi-class) ===")
logits = torch.tensor([[2.0, 1.0, 0.1],   # model predicts class 0
                        [0.1, 3.0, 0.2]])  # model predicts class 1
targets = torch.tensor([0, 1])              # correct classes

ce_loss = nn.CrossEntropyLoss()
loss = ce_loss(logits, targets)
print(f"  Logits: {logits.tolist()}")
print(f"  Targets: {targets.tolist()}")
print(f"  Loss: {loss.item():.4f}")
probs = F.softmax(logits, dim=1)
print(f"  Probabilities: {probs.tolist()}")
print(f"  Predicted classes: {logits.argmax(dim=1).tolist()}")

print("\n=== MSELoss (regression) ===")
predictions = torch.tensor([2.5, 0.0, 2.0, 8.0])
targets_reg = torch.tensor([3.0, -0.5, 2.0, 7.0])
mse_loss = nn.MSELoss()
print(f"  Predictions: {predictions.tolist()}")
print(f"  Targets:     {targets_reg.tolist()}")
print(f"  MSE Loss: {mse_loss(predictions, targets_reg).item():.4f}")

print("\n=== BCEWithLogitsLoss (multi-label) ===")
logits_ml = torch.tensor([[1.5, -0.5, 2.0]])  # one sample, 3 labels
targets_ml = torch.tensor([[1.0, 0.0, 1.0]])   # labels 0 and 2 are active
bce_loss = nn.BCEWithLogitsLoss()
print(f"  Logits: {logits_ml.tolist()}")
print(f"  Targets: {targets_ml.tolist()}")
print(f"  BCE Loss: {bce_loss(logits_ml, targets_ml).item():.4f}")

## 8. train() vs eval() Mode

Models behave differently during training and evaluation. Always set the right mode!
- `model.train()`: Dropout is active, BatchNorm uses batch statistics
- `model.eval()`: Dropout is disabled, BatchNorm uses running statistics

In [ ]:
model = SimpleCNN()
x = torch.randn(2, 1, 28, 28)

model.train()
out_train1 = model(x)
out_train2 = model(x)
print(f"Training mode: model.training={model.training}")
print(f"  Same input, two forward passes equal? {torch.equal(out_train1, out_train2)}")
print(f"  (Different because Dropout is random!)")

model.eval()
out_eval1 = model(x)
out_eval2 = model(x)
print(f"\nEval mode: model.training={model.training}")
print(f"  Same input, two forward passes equal? {torch.equal(out_eval1, out_eval2)}")
print(f"  (Identical because Dropout is disabled!)")

## 9. Saving and Loading Models

The recommended pattern: save and load the `state_dict` (not the whole model).

In [ ]:
import tempfile, os

model = SimpleCNN(num_classes=10)
x = torch.randn(1, 1, 28, 28)
original_output = model(x)

# Save
save_path = os.path.join(tempfile.gettempdir(), "model.pth")
torch.save(model.state_dict(), save_path)
print(f"Saved to {save_path} ({os.path.getsize(save_path):,} bytes)")

# Load into a NEW model instance
loaded_model = SimpleCNN(num_classes=10)  # must match architecture!
loaded_model.load_state_dict(torch.load(save_path, weights_only=True))
loaded_model.eval()

loaded_output = loaded_model(x)
print(f"Outputs match after load? {torch.allclose(original_output, loaded_output)}")

# Inspect what's in the state_dict
print(f"\nstate_dict keys ({len(model.state_dict())} entries):")
for key in list(model.state_dict().keys())[:6]:
    print(f"  {key}: {list(model.state_dict()[key].shape)}")
print(f"  ... and {len(model.state_dict()) - 6} more")

## 10. nn.Sequential vs Custom Forward

Use `nn.Sequential` for simple linear stacks. Use a custom `forward()` for anything with branching, residual connections, or dynamic behavior.

In [ ]:
# Quick Sequential model — great for simple architectures
quick_model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

print("Sequential model:")
print(quick_model)
print(f"\nOutput shape: {list(quick_model(torch.randn(4, 784)).shape)}")
print(f"Total params: {sum(p.numel() for p in quick_model.parameters()):,}")

## 🧪 Exercise: Build a 3-Layer CNN and Count Parameters

**Task:** Build a CNN with these specifications:
- Input: 3-channel 32x32 images (like CIFAR-10)
- Conv block 1: 3 → 32 channels, 3x3 kernel, padding=1, BatchNorm, ReLU, MaxPool 2x2
- Conv block 2: 32 → 64 channels, same pattern
- Conv block 3: 64 → 128 channels, same pattern
- Classifier: AdaptiveAvgPool2d(1) → Flatten → Linear(128, 10)

Count the total parameters and verify the output shape.

In [ ]:
def make_conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )

class CIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = make_conv_block(3, 32)
        self.block2 = make_conv_block(32, 64)
        self.block3 = make_conv_block(64, 128)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x)
        x = x.flatten(1)
        return self.classifier(x)

model = CIFAR10CNN()
x = torch.randn(8, 3, 32, 32)
out = model(x)
print(f"Input:  {list(x.shape)}")
print(f"Output: {list(out.shape)}")
print(f"\nParameter count by layer:")
for name, param in model.named_parameters():
    print(f"  {name:<35s} {str(list(param.shape)):>20s}  ({param.numel():>6,})")
print(f"\n  TOTAL: {sum(p.numel() for p in model.parameters()):,} parameters")

## Key Takeaways

1. **`nn.Module`** is the base class — define layers in `__init__`, logic in `forward()`
2. **Parameters** are learnable (optimized), **Buffers** are saved state (not optimized)
3. **`nn.Sequential`** for simple chains; custom `forward()` for complex architectures
4. **Always call `model.eval()`** before inference (disables Dropout, uses running BatchNorm stats)
5. **Save `state_dict()`**, not the whole model — more portable and explicit
6. **Layer choice**: Conv2d for spatial data, Linear for features, LayerNorm for transformers, BatchNorm for CNNs
7. **Activation choice**: ReLU for simplicity, GELU for transformers, SiLU for modern CNNs
8. **Loss choice**: CrossEntropy for classification, MSE for regression, BCEWithLogits for multi-label
9. **Count parameters** with `sum(p.numel() for p in model.parameters())`

**Next up:** Notebook 04 — The Complete Training Guide